# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

if api_key and api_key.startswith('AQ.') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gemini-3.1-flash-lite'
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=api_key)

API key looks good so far


In [4]:
links = fetch_website_links("https://anjoriarts.com")
links

['#main-content',
 '/',
 '/shop',
 '/categories',
 '/custom-order',
 '/about',
 '/blog',
 '/contact',
 '/wishlist',
 '/cart',
 '/login',
 '/shop',
 '/custom-order',
 '/shop',
 '/artworks/starry-night-crescent-moon-hand-painted-earrings',
 '/artworks/starry-night-crescent-moon-hand-painted-earrings',
 '/artworks/starry-night-crescent-moon-hand-painted-earrings',
 '/artworks/crimson-daisy-hand-painted-floral-earrings',
 '/artworks/crimson-daisy-hand-painted-floral-earrings',
 '/artworks/crimson-daisy-hand-painted-floral-earrings',
 '/artworks/nandi-the-divine-sovereign-in-gold-foil',
 '/artworks/nandi-the-divine-sovereign-in-gold-foil',
 '/artworks/nandi-the-divine-sovereign-in-gold-foil',
 '/artworks/chhath-puja-devotion-to-the-sun-god',
 '/artworks/chhath-puja-devotion-to-the-sun-god',
 '/artworks/chhath-puja-devotion-to-the-sun-god',
 '/custom-order',
 '/categories/madhubani',
 '/categories/tanjore',
 '/categories/mythological-devotional',
 '/categories',
 '/stories',
 '/reviews',
 'h

## First step: Have GPT-6 Luna figure out which links are relevant

### Use a call to GPT-6 Luna to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://anjoriarts.com"))


Here is the list of links on the website https://anjoriarts.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#main-content
/
/shop
/categories
/custom-order
/about
/blog
/contact
/wishlist
/cart
/login
/shop
/custom-order
/shop
/artworks/starry-night-crescent-moon-hand-painted-earrings
/artworks/starry-night-crescent-moon-hand-painted-earrings
/artworks/starry-night-crescent-moon-hand-painted-earrings
/artworks/crimson-daisy-hand-painted-floral-earrings
/artworks/crimson-daisy-hand-painted-floral-earrings
/artworks/crimson-daisy-hand-painted-floral-earrings
/artworks/nandi-the-divine-sovereign-in-gold-foil
/artworks/nandi-the-divine-sovereign-in-gold-foil
/artworks/nandi-the-divine-sovereign-in-gold-foil
/artworks/chhath-puja-devotion-to-the-sun-god
/artworks/chhath-puja-devotion-to-the-sun-god
/artwo

In [8]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [11]:
select_relevant_links("https://anjoriarts.com")

{'links': [{'type': 'about page', 'url': 'https://anjoriarts.com/about'},
  {'type': 'contact page', 'url': 'https://anjoriarts.com/contact'},
  {'type': 'stories page', 'url': 'https://anjoriarts.com/stories'},
  {'type': 'exhibitions page', 'url': 'https://anjoriarts.com/exhibitions'},
  {'type': 'reviews page', 'url': 'https://anjoriarts.com/reviews'},
  {'type': 'instagram page', 'url': 'https://www.instagram.com/anjoriarts'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/people/Anjori-Arts/61576973495407/'},
  {'type': 'youtube page', 'url': 'https://www.youtube.com/@anjoriarts'},
  {'type': 'pinterest page', 'url': 'https://in.pinterest.com/anjoriarts/'}]}

In [16]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [20]:
select_relevant_links("https://anjoriarts.com")

Selecting relevant links for https://anjoriarts.com by calling gemini-3.1-flash-lite
Found 10 relevant links


{'links': [{'type': 'home page', 'url': 'https://anjoriarts.com/'},
  {'type': 'about page', 'url': 'https://anjoriarts.com/about'},
  {'type': 'contact page', 'url': 'https://anjoriarts.com/contact'},
  {'type': 'blog page', 'url': 'https://anjoriarts.com/blog'},
  {'type': 'stories page', 'url': 'https://anjoriarts.com/stories'},
  {'type': 'exhibitions page', 'url': 'https://anjoriarts.com/exhibitions'},
  {'type': 'instagram', 'url': 'https://www.instagram.com/anjoriarts'},
  {'type': 'facebook',
   'url': 'https://www.facebook.com/people/Anjori-Arts/61576973495407/'},
  {'type': 'youtube', 'url': 'https://www.youtube.com/@anjoriarts'},
  {'type': 'pinterest', 'url': 'https://in.pinterest.com/anjoriarts/'}]}

In [21]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 8 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'brand assets', 'url': 'https://huggingface.co/brand'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'linkedin',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-6 Luna

In [22]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [23]:
print(fetch_page_and_all_relevant_links("https://anjoriarts.com"))

Selecting relevant links for https://anjoriarts.com by calling gemini-3.1-flash-lite
Found 10 relevant links
## Landing Page:

Anjori Arts | Authentic Indian Handmade Art & Custom Paintings

Skip to main content
Anjori Arts
Indian art, made by hand
Shop
Traditions
Custom Order
About
Blog
Contact
Search
Search artworks
Anjori Arts
/ Traditional art & custom orders
Art that carries a little more meaning.
Handmade Indian paintings and keepsakes, made with the patience of tradition and a place in your home in mind.
Shop artworks
Custom Order
Hand-painted originals
Made to your dimensions
A study in devotion & gold
Made to order
Your story, in colour.
Featured Artworks
Curated for your space.
View all artworks
Handcrafted Earrings
Starry Night Crescent Moon Hand-Painted Earrings
1" × 1" · Air dry clay
₹179
MRP
₹349
View
Handcrafted Earrings
Crimson Daisy Hand-Painted Floral Earrings
1" × 1" · Air dry clay
₹179
MRP
₹349
View
Tanjore
Nandi: The Divine Sovereign in Gold Foil
8" × 8" · Other
₹1

In [29]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [25]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [26]:
get_brochure_user_prompt("Anjori arts", "https://anjoriarts.com")

Selecting relevant links for https://anjoriarts.com by calling gemini-3.1-flash-lite
Found 10 relevant links


'\nYou are looking at a company called: Anjori arts\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nAnjori Arts | Authentic Indian Handmade Art & Custom Paintings\n\nSkip to main content\nAnjori Arts\nIndian art, made by hand\nShop\nTraditions\nCustom Order\nAbout\nBlog\nContact\nSearch\nSearch artworks\nAnjori Arts\n/ Traditional art & custom orders\nArt that carries a little more meaning.\nHandmade Indian paintings and keepsakes, made with the patience of tradition and a place in your home in mind.\nShop artworks\nCustom Order\nHand-painted originals\nMade to your dimensions\nA study in devotion & gold\nMade to order\nYour story, in colour.\nFeatured Artworks\nCurated for your space.\nView all artworks\nHandcrafted Earrings\nStarry Night Crescent Moon Hand-Painted Earrings\n1" × 1" · Air dry clay\n₹179\nMRP\n₹349\nView\nHandcrafted Earrings\nCr

In [30]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [31]:
create_brochure("Anjori arts", "https://anjoriarts.com")

Selecting relevant links for https://anjoriarts.com by calling gemini-3.1-flash-lite
Found 6 relevant links


# Anjori Arts: Where Tradition Meets "Wait, Did You Paint That With Your Hands?"

### Welcome to the Slow Lane
In a world of mass-produced, soul-crushing wall art that looks suspiciously like a printer error, Anjori Arts is here to save your walls (and your vibe). We believe in doing things the hard way—the traditional way. Founded by the talented Jyotsna Sharma, we specialize in art that requires patience, skill, and just the right amount of gold foil to make your living room look like a mythological palace.

### Our Vibe: Moonlight & Masterpieces
The name "Anjori" literally translates to "moonlight." We’re essentially the glow-up your home decor has been begging for. Whether it’s a 22K gold-foiled Tanjore masterpiece or a pair of air-dry clay earrings that make you look instantly cultured, our studio is where India’s heritage comes to hang out with modern design. 

**What we do when we aren’t drinking chai:**
* **Madhubani & Tanjore:** We keep the ancient traditions alive so you don't have to explain to guests why your wall looks like a history textbook—it just looks expensive and divine.
* **Custom Commissions:** Have a vision? We’ll paint it. We don’t believe in "printed reproductions." If you want your life story, your brand identity, or your portrait captured, we’ll do it line-by-line, layer-by-layer, until it’s perfect.
* **Handcrafted Accessories:** Because your ears deserve to wear art, too. 

### Why You’ll Love Us
* **No Robots Allowed:** Every single line is drawn by a human. We promise no printers were harmed (or used) in the making of your art.
* **Tailored to You:** We don't just dump art on your doorstep. We consult on surfaces, scale, and frames. We’re basically the interior design best friends you didn't know you needed.
* **The "Imperfection" Perfection:** We celebrate the slight, beautiful quirks of the handmade. It’s what proves it’s real—and honestly, it’s what gives it soul.

### Join the Cult(ure)
Are you a collector, a business owner looking for a branding identity that doesn’t look like a stock template, or just someone who appreciates things that take time? 

**The Careers Situation:** We are a team of artists, storytellers, and dreamers who prefer a brush to a keyboard. If you possess a steady hand, a deep respect for heritage, and an ungodly amount of patience for detail, keep an eye on our space. We’re always looking for people who understand that "fast" is rarely "fine."

### Let’s Get Personal
Ready to stop staring at that blank wall? 
* **Website:** [Anjori Arts](http://anjoriarts.com)
* **Email:** support@anjoriarts.com
* **WhatsApp/Call:** +91 80519 60916 (We’re here 7 days a week, because art never sleeps, and neither do we!)

*Anjori Arts: Your story, in color. And occasionally, in gold.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [32]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [33]:
stream_brochure("Anjori Arts", "https://anjoriarts.com")

Selecting relevant links for https://anjoriarts.com by calling gemini-3.1-flash-lite
Found 5 relevant links


# Welcome to Anjori Arts: Where Tradition Meets "Stop Looking at Your Phone and Look at This Painting"

Tired of your walls being as blank as your weekend plans? Welcome to **Anjori Arts**, where we turn "I need something to cover this paint stain" into a spiritual experience.

Founded by the visionary Jyotsna Sharma, we’re bridging the gap between centuries-old Indian heritage and your modern living room—all without a single soulless printer in sight. 

### Why Choose Us? (Besides the Obvious Good Taste)
*   **Zero Printing, 100% Sweat:** Everything is hand-painted. No mass-produced reproductions. If you see a line, a human being put it there with patience, focus, and—knowing artists—probably a lot of chai.
*   **Art That Actually Fits:** Whether you need a 22K gold-foiled Tanjore masterpiece or custom branding that says "I’m professional but I have a soul," we create it specifically for your space and story.
*   **The "Anjori" Glow-Up:** Our name means "moonlight," which is essentially what your home is missing. We bring the divine radiance of the Sun and Moon to your decor without the pesky solar flares.

### Our Vibe
We are "devotion meets dimension." From intricate **Madhubani** geometric patterns to **Tanjore** gold-foil work that screams "I’m fancy, but I respect the ancestors," we do it all. We also make air-dry clay earrings, so you can literally wear our art to dinner and humble-brag about it.

### Who Are You?
*   **The Collector:** You want a house that looks like a storybook, not a hotel lobby.
*   **The Business Owner:** You’re tired of corporate logos that look like they were made in a blender. Let’s give your brand some actual personality.
*   **The Gift-Giver:** You want to be the person who gives the "most thoughtful gift" award. (You’re welcome.)

### Join the Chaos (or the Culture)
Are you a creative soul who prefers a paintbrush to a spreadsheet? While we’re currently busy painting history into existence, keep an eye on our progress. We value the "slight imperfections of the handmade"—which is corporate-speak for "we love authentic, human, non-robot energy." 

**Want to chat?** 
Our team responds within 24 hours. We’re open 7 days a week, because art (and your desire to fill that empty corner) doesn't take a day off.

**Get in touch at:**
*   **Email:** support@anjoriarts.com
*   **WhatsApp/Phone:** +91 80519 60916
*   **Website:** [anjoriarts.com](http://anjoriarts.com)

**Anjori Arts: Because your walls deserve better than a beige coat of paint and a generic framed quote.**

In [34]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 7 relevant links


# Welcome to Hugging Face: Where the Robots Come to Play

Ever wondered where the world’s AI developers go when they want to build the future, share their digital breakthroughs, and occasionally argue about which transformer architecture is better? You’ve arrived. 

Welcome to **Hugging Face**, the open-source playground that has become the beating heart of the machine learning revolution.

### What is this place?
Think of us as the GitHub for AI, but with more emoji and significantly less "It worked on my machine" heartbreak. We provide the infrastructure for the machine learning community to host, share, and collaborate on over 2 million models, half a million datasets, and a million-plus AI applications. 

Whether you’re training a Mixture-of-Experts model, turning text into videos, or just looking for a really good linter, we’ve got the tools—and the community—to make it happen.

### Our Culture: Serious Tech, Serious Fun
At Hugging Face, we believe the future of AI shouldn’t be locked behind a boardroom door. We are the champions of the Open-Source ecosystem. Our culture is defined by:
* **Radical Collaboration:** If someone builds a better way to tokenize data, we share it. If there’s a bug, we fix it together. 
* **The "Hugging" Spirit:** We take our technology seriously, but we don't take ourselves seriously. We’re here to build, learn, and democratize the power of intelligence.
* **Continuous Learning:** From our daily papers to our deep-dive technical blogs, we’re obsessed with staying on the bleeding edge.

### Who uses Hugging Face?
Honestly? Everyone. From solo developers tinkering in their basements to enterprise giants looking to deploy robust inference endpoints, we’re the industry standard for hosting models. If it involves machine learning, someone on our platform is likely already pushing the boundaries of what’s possible with it.

### Careers: Join the Team Building the Future
Do you love machine learning, scalability, and being part of the most vibrant technical community on the planet? We’re always looking for people who want to:
* **Empower the Community:** Build the infrastructure that millions of developers rely on every day.
* **Push the Limits:** Work on the latest in agents, hardware acceleration, and open-source research.
* **Shape the Landscape:** Influence how AI is developed, distributed, and understood globally.

If you enjoy working at the intersection of high-scale engineering and collaborative innovation, come help us turn "Token-In, Token-Out" into a global standard. 

### Quick Stats for the Curious
* **2M+ Models:** Find anything from coding assistants to experimental art generators.
* **1M+ Apps:** Explore our Spaces to see models live in action.
* **110,000+ Followers:** Join the largest AI conversation on the internet.

**Ready to start?**
Browse our models, join the Discord, or start your first repository. The AI revolution isn't coming—it's being pushed to the hub right now.

*Hugging Face: The AI community building the future. (And yes, we really like the emoji.)*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>